# Case Study - Customer Churn

Predicting whether a customer will churn based on their tenure, monthly charges, and contract type using classification models.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
%matplotlib inline

In [2]:
np.random.seed(42)
n = 1000

tenure = np.random.randint(1, 73, n)
monthly_charges = np.round(np.random.uniform(20, 150, n), 2)
contract_type = np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2])

data = pd.DataFrame({
    'tenure': tenure,
    'monthly_charges': monthly_charges,
    'contract_type': contract_type
})

data['churn'] = 0
high_risk = (data['tenure'] < 12) | (data['contract_type'] == 'Month-to-month')
data.loc[high_risk, 'churn'] = np.random.binomial(1, 0.45, high_risk.sum())
data.loc[~high_risk, 'churn'] = np.random.binomial(1, 0.08, (~high_risk).sum())

print ('Data shape: %s\n' % str(data.shape))
print (data.head())

Data shape: (1000, 4)
   tenure  monthly_charges   contract_type  churn
0      51          119.14     Two year      0
1      67           54.53  Month-to-month  1
2      26           65.76      One year    0
3      26           60.38      One year    0
4      38          125.55  Month-to-month  1


In [3]:
print ('Dataset statistics:\n')
print (data.describe())

       tenure  monthly_charges      churn
count  1000.00      1000.00     1000.00
mean     36.37        84.62        0.27
std      20.82        37.45        0.44
min       1.00        20.06        0.00
25%      18.00        52.09        0.00
50%      36.00        85.31        0.00
75%      54.00       117.45        1.00
max      72.00       149.97        1.00


In [4]:
print ('Churn distribution:\n')
print (data['churn'].value_counts())

Churn distribution:
0    734
1    266
Name: churn, dtype: int64


In [5]:
print ('Contract type distribution:\n')
print (data['contract_type'].value_counts())

Contract type distribution:
Month-to-month    506
One year          298
Two year          196
Name: contract_type, dtype: int64


In [6]:
print ('Churn rate by contract type:\n')
print (data.groupby('contract_type')['churn'].mean())

Churn rate by contract type:
contract_type
Month-to-month    0.42
One year          0.11
Two year          0.07
Name: churn, dtype: float64


<hr>

In [7]:
le = LabelEncoder()
data['contract_encoded'] = le.fit_transform(data['contract_type'])

In [8]:
X = data[['tenure', 'monthly_charges', 'contract_encoded']]
y = data['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print ('Train size: %d, Test size: %d\n' % (len(X_train), len(X_test)))
print ('Train churn rate: %.2f, Test churn rate: %.2f\n' % (y_train.mean(), y_test.mean()))

Train size: 700, Test size: 300
Train churn rate: 0.27, Test churn rate: 0.26


In [9]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

lr_train_acc = accuracy_score(y_train, lr.predict(X_train))
lr_test_acc = accuracy_score(y_test, lr.predict(X_test))

print ('LogisticRegression - Train Accuracy: %.2f\n' % lr_train_acc)
print ('LogisticRegression - Test Accuracy: %.2f\n' % lr_test_acc)

LogisticRegression - Train Accuracy: 0.80
LogisticRegression - Test Accuracy: 0.78


In [10]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))

print ('RandomForestClassifier - Train Accuracy: %.2f\n' % rf_train_acc)
print ('RandomForestClassifier - Test Accuracy: %.2f\n' % rf_test_acc)

RandomForestClassifier - Train Accuracy: 0.99
RandomForestClassifier - Test Accuracy: 0.80


<hr>

In [11]:
lr_cm = confusion_matrix(y_test, lr.predict(X_test))
rf_cm = confusion_matrix(y_test, rf.predict(X_test))

print ('=== LogisticRegression Confusion Matrix ===\n')
print (lr_cm)
print ('\n')
print ('=== RandomForestClassifier Confusion Matrix ===\n')
print (rf_cm)

=== LogisticRegression Confusion Matrix ===
[[197  21]
 [ 45  37]]

=== RandomForestClassifier Confusion Matrix ===
[[199  19]
 [ 41  41]]


In [12]:
print ('=== LogisticRegression Classification Report ===\n')
print (classification_report(y_test, lr.predict(X_test)))
print ('=== RandomForestClassifier Classification Report ===\n')
print (classification_report(y_test, rf.predict(X_test)))

=== LogisticRegression Classification Report ===
              precision    recall  f1-score   support

           0       0.81      0.90      0.86       218
           1       0.64      0.45      0.53        82

    accuracy                           0.78       300
   macro avg       0.73      0.68      0.69       300
weighted avg       0.77      0.78      0.77       300


=== RandomForestClassifier Classification Report ===
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       218
           1       0.68      0.50      0.58        82

    accuracy                           0.80       300
   macro avg       0.76      0.71      0.72       300
weighted avg       0.79      0.80      0.79       300
